In [10]:
import os
import sys
import copy
import pandas as pd
import numpy as np

# get the absolute path of the current file
current_dir = os.path.dirname(os.path.abspath("__file__"))
# get the project root (one level up from notebooks/)
project_root = os.path.abspath(os.path.join(current_dir, ".."))
# add project root to sys.path
if project_root not in sys.path:
    sys.path.append(project_root)

In [11]:
from utils.stock_utils import download_stock_data
from utils.kpi import CAGR, volatility, Sharpe, max_dd, Sortino, calamar

List of all the stocks I want to have in my tradable universe for backtesting

In [12]:
mid_cap = ["PAYTM.NS", "NYKAA.NS", "VMM.NS", "MFSL.NS", "EXIDEIND.NS", "UPL.NS", "BSE.NS", "POLICYBZR.NS", "M&MFIN.NS", "MANKIND.NS", "NATIONALUM.NS", "ABFRL.NS", "ESCORTS.NS", "WAAREEENER.NS", "TIINDIA.NS", "NMDC.NS", "KALYANKJIL.NS", "SUZLON.NS", "UNIONBANK.NS", "SRF.NS"]
# mid_cap = ["PAYTM.NS"]

# PORTFOLIO REBALANCING

In [13]:
mid_cap_dict = download_stock_data(mid_cap, 500, "1mo")

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********

In [14]:
def get_monthly_return(mid_cap_dict):
    """
    Calculate monthly returns for the given mid-cap stocks.
    
    Parameters:
    mid_cap_dict (dictionary): Dictionary of stock symbols.
    
    Returns:
    pd.DataFrame: DataFrame containing monthly returns for each stock.
    """
    ohlc_dict = copy.deepcopy(mid_cap_dict)
    return_df = pd.DataFrame()
    for stock in mid_cap:
        print("calculated monthly return for ", stock)
        ohlc_dict[stock]["mon_ret"] = ohlc_dict[stock]["Close"].pct_change()
        return_df[stock] = ohlc_dict[stock]["mon_ret"]
    return_df.dropna(inplace=True)
    return return_df

In [15]:
return_df = get_monthly_return(mid_cap_dict)

calculated monthly return for  PAYTM.NS
calculated monthly return for  NYKAA.NS
calculated monthly return for  VMM.NS
calculated monthly return for  MFSL.NS
calculated monthly return for  EXIDEIND.NS
calculated monthly return for  UPL.NS
calculated monthly return for  BSE.NS
calculated monthly return for  POLICYBZR.NS
calculated monthly return for  M&MFIN.NS
calculated monthly return for  MANKIND.NS
calculated monthly return for  NATIONALUM.NS
calculated monthly return for  ABFRL.NS
calculated monthly return for  ESCORTS.NS
calculated monthly return for  WAAREEENER.NS
calculated monthly return for  TIINDIA.NS
calculated monthly return for  NMDC.NS
calculated monthly return for  KALYANKJIL.NS
calculated monthly return for  SUZLON.NS
calculated monthly return for  UNIONBANK.NS
calculated monthly return for  SRF.NS


In [17]:
def portfolio_rebalance(return_df, keep, throw):
    """
    Rebalance the portfolio based on monthly returns.
    
    Parameters:
    return_df (pd.DataFrame): DataFrame containing monthly returns for each stock.
    keep (int): Number of stocks to keep in the portfolio.
    throw (int): Number of stocks to throw out of the portfolio.
    initial_investment (float): Initial investment amount (default is 1,000,000).
    
    Returns:
    pd.DataFrame: DataFrame containing portfolio value over time.
    """
    df = return_df.copy()
    portfolio = []
    monthly_return = []
    for i in range (len(df)):
        if len(portfolio) > 0:
            monthly_return.append(df[portfolio].iloc[i, :].mean())
            bad_stocks = df[portfolio].iloc[i,:].sort_values(ascending=True)[:throw].index.values.tolist()
            portfolio = [t for t in portfolio if t not in bad_stocks]
        else:
            monthly_return.append(0)
            print("No stocks in portfolio")
        fill = keep - len(portfolio)
        new_picks = df.iloc[i,:].sort_values(ascending=False)[:fill].index.values.tolist()
        portfolio = portfolio + new_picks
        print(portfolio)
    monthly_ret_df = pd.DataFrame(np.array(monthly_return),columns=["mon_ret"])
    return monthly_ret_df

In [18]:
new_portfolio = portfolio_rebalance(return_df, 6, 3)
CAGR(new_portfolio, 12, "mon_ret", 1)  # Calculate CAGR

No stocks in portfolio
['SRF.NS', 'UPL.NS', 'ESCORTS.NS', 'M&MFIN.NS', 'NYKAA.NS', 'VMM.NS']
['SRF.NS', 'UPL.NS', 'M&MFIN.NS', 'UPL.NS', 'SRF.NS', 'UNIONBANK.NS']
['SRF.NS', 'SRF.NS', 'UNIONBANK.NS', 'BSE.NS', 'MFSL.NS', 'SUZLON.NS']
['BSE.NS', 'MFSL.NS', 'BSE.NS', 'MFSL.NS', 'VMM.NS', 'KALYANKJIL.NS']
['BSE.NS', 'BSE.NS', 'SUZLON.NS', 'BSE.NS', 'UNIONBANK.NS', 'NATIONALUM.NS']
['UNIONBANK.NS', 'NATIONALUM.NS', 'SRF.NS', 'MFSL.NS', 'VMM.NS', 'NATIONALUM.NS']
['NATIONALUM.NS', 'VMM.NS', 'NATIONALUM.NS', 'PAYTM.NS', 'MANKIND.NS', 'KALYANKJIL.NS']
['VMM.NS', 'PAYTM.NS', 'WAAREEENER.NS', 'PAYTM.NS', 'NYKAA.NS', 'VMM.NS']
0    0.000000
1   -0.057811
2    0.047879
3    0.056201
4    0.160513
5    0.028347
6   -0.055957
7    0.004677
Name: return, dtype: float64


np.float64(0.28234951535241537)

In [19]:
volatility(new_portfolio, 12, "mon_ret")  # Calculate Volatility

np.float64(0.22678919861919125)

In [ ]:
Sharpe(new_portfolio, "mon_re", True)  # Calculate Sharpe Ratio

KeyError: 'Close'